<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/15_chunk_overlap_rag/chunk_overlap_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
document = """
Paris is the capital city of France. It is known for culture and tourism.
France is located in Europe and has historical importance.
The Eiffel Tower is located in Paris and attracts millions of visitors.
"""

In [9]:
def chunk_text_with_overlap(text, chunk_size=20, overlap=5):
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap

    return chunks

chunks = chunk_text_with_overlap(document)

print("CHUNKS WITH OVERLAP:\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}\n")

CHUNKS WITH OVERLAP:

Chunk 1: Paris is the capital city of France. It is known for culture and tourism. France is located in Europe and

Chunk 2: is located in Europe and has historical importance. The Eiffel Tower is located in Paris and attracts millions of visitors.

Chunk 3: and attracts millions of visitors.



In [10]:
import faiss
import numpy as np

embeddings = embed_model.encode(chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [11]:
query = "Where is the Eiffel Tower located?"

query_embedding = embed_model.encode([query])

In [12]:
D, I = index.search(np.array(query_embedding), k=1)

retrieved_chunk = chunks[I[0][0]]

print("RETRIEVED CHUNK:")
print(retrieved_chunk)

RETRIEVED CHUNK:
is located in Europe and has historical importance. The Eiffel Tower is located in Paris and attracts millions of visitors.


In [13]:
prompt = f"""
Use the following context to answer the question.

Context: {retrieved_chunk}

Question: {query}
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=30)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
Paris
